# SKANN-SSL V3 Training - Kaggle T4x2

**Batch Size Experiment:** batch=4 per GPU (effective batch=8 with DDP)

## Hypothesis
Smaller batch → noisier BT gradients → better fine-grained class separation (fishing vs small_craft)

## Setup
1. Upload your dataset to Kaggle as `skann-ssl-v3-data`
2. Add it to this notebook via "Add Data"
3. Enable GPU T4x2 in notebook settings

## Cell 0: Setup & Data Extraction

In [ ]:
import os
import zipfile
import torch

print("="*60)
print("SKANN-SSL V3 - Kaggle T4x2 Setup")
print("="*60)

# Check GPUs
n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB")

# Find dataset
INPUT_DIR = "/kaggle/input"
dataset_dirs = [d for d in os.listdir(INPUT_DIR) if 'skann' in d.lower()]
if dataset_dirs:
    DATA_SOURCE = os.path.join(INPUT_DIR, dataset_dirs[0])
    print(f"\nFound dataset: {DATA_SOURCE}")
else:
    # Fallback - list what's available
    print(f"\nAvailable in {INPUT_DIR}:")
    for d in os.listdir(INPUT_DIR):
        print(f"  {d}")
    DATA_SOURCE = os.path.join(INPUT_DIR, os.listdir(INPUT_DIR)[0])
    print(f"Using: {DATA_SOURCE}")

print(f"\nContents of {DATA_SOURCE}:")
for f in os.listdir(DATA_SOURCE):
    fpath = os.path.join(DATA_SOURCE, f)
    size = os.path.getsize(fpath) / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Extract tensors if needed
WORK_DIR = "/kaggle/working/data"
os.makedirs(WORK_DIR, exist_ok=True)

# Find and extract zip
zip_files = [f for f in os.listdir(DATA_SOURCE) if f.endswith('.zip')]
if zip_files:
    zip_path = os.path.join(DATA_SOURCE, zip_files[0])
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(WORK_DIR)
    print("✅ Extraction complete")

# Find CSVs
csv_files = [f for f in os.listdir(DATA_SOURCE) if f.endswith('.csv')]
for csv in csv_files:
    src = os.path.join(DATA_SOURCE, csv)
    dst = os.path.join(WORK_DIR, csv)
    if not os.path.exists(dst):
        import shutil
        shutil.copy(src, dst)
        print(f"Copied {csv}")

# Identify files
import pandas as pd

csv_paths = [os.path.join(WORK_DIR, f) for f in os.listdir(WORK_DIR) if f.endswith('.csv')]
for p in csv_paths:
    df = pd.read_csv(p)
    print(f"\n{os.path.basename(p)}: {len(df)} rows")
    print(f"  Columns: {list(df.columns)[:5]}...")
    if 'anchor_clip_id' in df.columns:
        PAIRING_PATH = p
        print("  → PAIRING file")
    elif 'clip_id' in df.columns:
        MANIFEST_PATH = p
        print("  → MANIFEST file")

# Find tensor directory
for root, dirs, files in os.walk(WORK_DIR):
    npy_count = len([f for f in files if f.endswith('.npy')])
    if npy_count > 100:
        TENSOR_DIR = root
        print(f"\nTensor directory: {TENSOR_DIR}")
        print(f"  Found {npy_count} .npy files")
        break

print("\n" + "="*60)
print("Paths configured:")
print(f"  MANIFEST_PATH: {MANIFEST_PATH}")
print(f"  PAIRING_PATH:  {PAIRING_PATH}")
print(f"  TENSOR_DIR:    {TENSOR_DIR}")
print("="*60)

## Cell 1: Configuration

In [ ]:
import time
from datetime import datetime

def log(msg):
    ts = time.strftime('%H:%M:%S')
    print(f"[{ts}] {msg}", flush=True)

# =============================================================================
# PATHS
# =============================================================================
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# HYPERPARAMETERS - BATCH=2 FOR NOISY BT GRADIENTS
# =============================================================================
EPOCHS = 50
BATCH_SIZE = 2              # Per GPU! Effective = 8 with 2 GPUs
BASE_LR = 1e-4
WEIGHT_DECAY = 0.01
BT_LAMBDA = 5e-3
LATENT_DIM = 256

# Logging
LOG_EVERY = 1                # Every epoch
HEALTH_CHECK_EVERY = 5       # Health metrics every 5 epochs
CHECKPOINT_EVERY = 5

# DDP settings
USE_DDP = torch.cuda.device_count() > 1

print("="*60)
print("SKANN-SSL V3 Configuration (Kaggle T4x2)")
print("="*60)
print(f"Batch size per GPU: {BATCH_SIZE}")
print(f"Effective batch: {BATCH_SIZE * max(1, torch.cuda.device_count())}")
print(f"DDP enabled: {USE_DDP}")
print(f"Latent dim: {LATENT_DIM}")
print(f"SK Kernels: (31, 63, 127, 255, 511, 1023)")
print(f"Projector: 512 → 4096 → 8192 → 16384 → {LATENT_DIM}")
print(f"Health check every: {HEALTH_CHECK_EVERY} epochs")
print("="*60)

## Cell 2: Validate Data

In [ ]:
import pandas as pd
import numpy as np

assert os.path.exists(PAIRING_PATH), f"Pairing not found: {PAIRING_PATH}"
assert os.path.exists(TENSOR_DIR), f"Tensors not found: {TENSOR_DIR}"

pairing_df = pd.read_csv(PAIRING_PATH)
log(f"Pairing manifest: {len(pairing_df)} entries")

print(f"\nClass distribution:")
for cls, count in pairing_df['vessel_class'].value_counts().items():
    print(f"  {cls}: {count}")

tensor_files = [f for f in os.listdir(TENSOR_DIR) if f.endswith('.npy')]
log(f"Tensor files: {len(tensor_files)}")

sample = np.load(os.path.join(TENSOR_DIR, tensor_files[0]))
log(f"Tensor shape: {sample.shape} ({sample.size/16000:.1f} sec @ 16kHz)")

## Cell 3: SKANN Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# SKANN: SELECTIVE KERNEL AUDIO NEURAL NETWORKS (V2.1.0)
# SK Kernels: (31, 63, 127, 255, 511, 1023)
# =============================================================================

def _norm_1d(channels, kind='gn', groups=8):
    if kind == 'bn':
        return nn.BatchNorm1d(channels)
    if kind == 'ln':
        return nn.GroupNorm(1, channels)
    return nn.GroupNorm(min(groups, channels), channels)


class SKConv1D(nn.Module):
    """Selective Kernel 1D Convolution - V2.1.0"""
    
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),
        stride: int = 1,
        reduction: int = 16,
        norm: str = 'gn',
        act: str = 'gelu',
        residual: bool = True,
        dropout: float = 0.0
    ):
        super().__init__()
        
        self.branches = nn.ModuleList()
        for k in kernel_sizes:
            pad = k // 2
            self.branches.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=stride, 
                         padding=pad, bias=False)
            )
        
        self.n_branches = len(kernel_sizes)
        self.out_ch = out_ch
        self.kernel_sizes = kernel_sizes
        
        hidden = max(out_ch // reduction, 8)
        self.fc1 = nn.Linear(out_ch, hidden)
        self.fc2 = nn.Linear(hidden, out_ch * self.n_branches)
        
        self.norm = _norm_1d(out_ch, norm)
        self.act = nn.GELU() if act == 'gelu' else nn.ReLU(inplace=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        self.residual = residual
        self.match = None
        if residual and (in_ch != out_ch or stride != 1):
            self.match = nn.Conv1d(in_ch, out_ch, kernel_size=1, 
                                   stride=stride, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = [branch(x) for branch in self.branches]
        U = torch.stack(feats, dim=1).sum(dim=1)
        s = F.adaptive_avg_pool1d(U, 1).squeeze(-1)
        z = self.fc2(F.relu(self.fc1(s), inplace=False))
        a = z.view(z.size(0), self.n_branches, self.out_ch)
        a = F.softmax(a, dim=1).unsqueeze(-1)
        feats_stacked = torch.stack(feats, dim=1)
        V = (a * feats_stacked).sum(dim=1)
        out = self.norm(V)
        out = self.act(out)
        out = self.dropout(out)
        if self.residual:
            res = x if self.match is None else self.match(x)
            out = out + res
        return out


class SKFilterbank(nn.Module):
    """Selective Kernel Filterbank - V2.1.0"""
    
    def __init__(
        self,
        out_ch: int = 64,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),
        norm: str = 'gn'
    ):
        super().__init__()
        
        self.stem = SKConv1D(
            in_ch=1,
            out_ch=out_ch,
            kernel_sizes=kernel_sizes,
            stride=1,
            reduction=16,
            norm=norm,
            act='gelu',
            residual=False
        )
        self.post_norm = _norm_1d(out_ch, norm)
        
        print(f"    SKFilterbank kernels: {kernel_sizes}")
        print(f"    Frequency coverage @ 16kHz: {16000/kernel_sizes[-1]:.0f}Hz - {16000/kernel_sizes[0]:.0f}Hz")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        return self.post_norm(h)


class HybridSKEncoderV3(nn.Module):
    """SKANN-SSL V3 Encoder with return_features option"""
    
    def __init__(self, latent_dim=256):
        super().__init__()
        
        print("\n" + "="*60)
        print("Building SKANN-SSL V3 Encoder")
        print("="*60)
        print("  Base: V2.1.0 architecture")
        
        # SK Frontend
        self.sk_frontend = SKFilterbank(
            out_ch=64,
            kernel_sizes=(31, 63, 127, 255, 511, 1023)
        )
        self.downsample = nn.AvgPool1d(kernel_size=8, stride=8)
        
        # Channel bridge
        self.channel_bridge = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=False)
        )
        
        # 2D Backbone → h (512-dim)
        self.backbone2d = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.Conv2d(64, 128, 3, padding=1, stride=(2, 2)),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.Conv2d(128, 256, 3, padding=1, stride=(2, 1)),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1)
        )
        
        # Projector: 512 → 4096 → 8192 → 16384 → 256
        self.projector = nn.Sequential(
            nn.Linear(512, 4096),
            nn.LayerNorm(4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, 8192),
            nn.LayerNorm(8192),
            nn.ReLU(inplace=False),
            nn.Linear(8192, 16384),
            nn.LayerNorm(16384),
            nn.ReLU(inplace=False),
            nn.Linear(16384, latent_dim)
        )
        
        print(f"    Backbone output (h): 512-dim")
        print(f"    Projector: 512 → 4096 → 8192 → 16384 → {latent_dim}")
        self._count_params()
        print("="*60 + "\n")
    
    def _count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.1f}M")
    
    def forward(self, x, return_features=False):
        if x.dim() > 3:
            x = x.view(x.size(0), -1).unsqueeze(1)
        elif x.dim() == 2:
            x = x.unsqueeze(1)
        
        x = self.sk_frontend(x)
        x = self.downsample(x)
        x = self.channel_bridge(x)
        x = x.unsqueeze(1)
        
        h = self.backbone2d(x)
        z = self.projector(h)
        
        if return_features:
            return h, z
        return z


# Test
model = HybridSKEncoderV3(latent_dim=LATENT_DIM)
dummy = torch.randn(2, 1, 80000)
h, z = model(dummy, return_features=True)
print(f"✅ Test: Input {dummy.shape} → h {h.shape}, z {z.shape}")

## Cell 4: Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class HierarchicalDataset(Dataset):
    def __init__(self, pairing_path, data_dir):
        self.df = pd.read_csv(pairing_path)
        self.data_dir = data_dir
        self.class_to_id = {c: i for i, c in enumerate(sorted(self.df["vessel_class"].unique()))}
        print(f"    Classes: {list(self.class_to_id.keys())}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        anchor_id = int(row['anchor_clip_id'])
        p_ids = str(row['partner_clip_ids']).split('|')
        p_ids = [int(x) for x in p_ids if x.strip()]
        partner_id = np.random.choice(p_ids)
        
        y1 = np.load(os.path.join(self.data_dir, f"tensor_{anchor_id:06d}.npy")).flatten()
        y2 = np.load(os.path.join(self.data_dir, f"tensor_{partner_id:06d}.npy")).flatten()
        
        return (
            torch.from_numpy(y1).float(),
            torch.from_numpy(y2).float(),
            self.class_to_id[row["vessel_class"]]
        )


dataset = HierarchicalDataset(PAIRING_PATH, TENSOR_DIR)
log(f"Dataset: {len(dataset)} samples")

## Cell 5: Barlow Twins Loss & Health Metrics

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score


def barlow_twins_loss(z1, z2, lambd=BT_LAMBDA):
    """Barlow Twins loss"""
    batch_size = z1.size(0)
    
    z1_mean = z1.mean(dim=0)
    z1_std = z1.std(dim=0) + 1e-6
    z1_norm = (z1 - z1_mean) / z1_std
    
    z2_mean = z2.mean(dim=0)
    z2_std = z2.std(dim=0) + 1e-6
    z2_norm = (z2 - z2_mean) / z2_std
    
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    
    diag = torch.diagonal(c)
    on_diag_loss = torch.pow(1.0 - diag, 2).sum()
    c_squared = torch.pow(c, 2)
    off_diag_loss = c_squared.sum() - torch.pow(diag, 2).sum()
    
    return on_diag_loss + lambd * off_diag_loss


def create_eval_subset(pairing_path, n_per_class=200):
    """Create balanced subset for evaluation."""
    df = pd.read_csv(pairing_path)
    classes = sorted(df['vessel_class'].unique())
    eval_indices = []
    for cls in classes:
        cls_df = df[df['vessel_class'] == cls]
        sampled = cls_df.sample(n=min(n_per_class, len(cls_df)), random_state=42)
        eval_indices.extend(sampled.index.tolist())
    return eval_indices


@torch.no_grad()
def compute_health_metrics(model, eval_indices, pairing_df, tensor_dir, class_to_id, device):
    """SSL Health Metrics on backbone (h) and projector (z)"""
    # Handle DDP model
    m = model.module if hasattr(model, 'module') else model
    m.eval()
    
    h_embeddings, z_embeddings, labels = [], [], []
    
    for idx in eval_indices:
        row = pairing_df.iloc[idx]
        clip_id = int(row['anchor_clip_id']) if 'anchor_clip_id' in row.index else int(row['clip_id'])
        label = class_to_id[row['vessel_class']]
        
        tensor_path = os.path.join(tensor_dir, f"tensor_{clip_id:06d}.npy")
        x = np.load(tensor_path).astype(np.float32).flatten()
        x = torch.from_numpy(x).unsqueeze(0).to(device)
        
        h, z = m(x, return_features=True)
        h_embeddings.append(h.cpu().numpy().squeeze())
        z_embeddings.append(z.cpu().numpy().squeeze())
        labels.append(label)
    
    h_embeddings = np.array(h_embeddings)
    z_embeddings = np.array(z_embeddings)
    labels = np.array(labels)
    
    sil_h = silhouette_score(h_embeddings, labels, metric='cosine')
    sil_z = silhouette_score(z_embeddings, labels, metric='cosine')
    
    sil_h_samples = silhouette_samples(h_embeddings, labels, metric='cosine')
    class_names = sorted(class_to_id.keys())
    per_class_h = {}
    for i, cls in enumerate(class_names):
        mask = labels == i
        if mask.sum() > 0:
            per_class_h[cls] = float(sil_h_samples[mask].mean())
    
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn_scores = cross_val_score(knn, h_embeddings, labels, cv=5, scoring='accuracy')
    knn_acc = knn_scores.mean()
    
    m.train()
    
    return {
        'sil_h': sil_h,
        'sil_z': sil_z,
        'knn_acc': knn_acc,
        'per_class_h': per_class_h
    }

## Cell 6: Training Loop (Single GPU or DDP)

In [ ]:
from tqdm.notebook import tqdm
import glob

def train():
    log("Starting SKANN-SSL V3 Training (Kaggle)")
    
    n_gpus = torch.cuda.device_count()
    effective_batch = BATCH_SIZE * max(1, n_gpus)
    
    print("\n" + "="*60)
    print("SKANN-SSL V3 Training - BATCH=4 EXPERIMENT")
    print("="*60)
    print(f"  SK Kernels: (31, 63, 127, 255, 511, 1023)")
    print(f"  Projector: 512 → 4096 → 8192 → 16384 → {LATENT_DIM}")
    print(f"  Batch size per GPU: {BATCH_SIZE}")
    print(f"  GPUs: {n_gpus}")
    print(f"  Effective batch: {effective_batch}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Health check every: {HEALTH_CHECK_EVERY} epochs")
    print("="*60 + "\n")
    
    device = torch.device('cuda:0')
    log(f"Primary device: {device}")
    
    # Model
    model = HybridSKEncoderV3(latent_dim=LATENT_DIM).to(device)
    
    # DataParallel for multi-GPU (simpler than DDP for Kaggle)
    if n_gpus > 1:
        log(f"Using DataParallel with {n_gpus} GPUs")
        model = nn.DataParallel(model)
    
    # Dataset
    dataset = HierarchicalDataset(PAIRING_PATH, TENSOR_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE * max(1, n_gpus),  # Scale batch with GPUs
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    log(f"Batches per epoch: {len(loader)}")
    
    # Evaluation setup
    pairing_df = pd.read_csv(PAIRING_PATH)
    class_to_id = {c: i for i, c in enumerate(sorted(pairing_df['vessel_class'].unique()))}
    class_names = sorted(class_to_id.keys())
    eval_indices = create_eval_subset(PAIRING_PATH, n_per_class=200)
    log(f"Eval subset: {len(eval_indices)} clips")
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # Mixed precision for T4
    scaler = torch.cuda.amp.GradScaler()
    
    # CSV logging
    loss_csv = os.path.join(OUTPUT_DIR, 'loss_history.csv')
    header = "epoch,loss,lr,sil_h,sil_z,knn_acc," + ",".join([f"sil_h_{c}" for c in class_names])
    with open(loss_csv, 'w') as f:
        f.write(header + "\n")
    log(f"Loss history: {loss_csv}")
    
    best_sil_h = -1.0
    start_time = time.time()
    
    log(f"Starting training...\n")
    
    for epoch in tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch"):
        model.train()
        total_loss = 0.0
        
        for y1, y2, _ in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            y1 = y1.to(device, non_blocking=True)
            y2 = y2.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Mixed precision
            with torch.cuda.amp.autocast():
                z1 = model(y1)
                z2 = model(y2)
                loss = barlow_twins_loss(z1, z2)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(loader)
        current_lr = scheduler.get_last_lr()[0]
        
        # Health metrics
        metrics = None
        if epoch % HEALTH_CHECK_EVERY == 0:
            metrics = compute_health_metrics(
                model, eval_indices, pairing_df, TENSOR_DIR, class_to_id, device
            )
            
            if metrics['sil_h'] > best_sil_h:
                best_sil_h = metrics['sil_h']
                best_path = os.path.join(OUTPUT_DIR, "best_model.pth")
                state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
                torch.save(state, best_path)
                tqdm.write(f"  🏆 New best sil_h: {metrics['sil_h']:.4f}")
            
            if metrics['sil_h'] > 0.5 and metrics['sil_z'] < 0.3:
                tqdm.write(f"  ⚠️ PROJECTOR ISSUE: sil_h={metrics['sil_h']:.4f} but sil_z={metrics['sil_z']:.4f}")
        
        # Logging
        if epoch % LOG_EVERY == 0 or metrics is not None:
            elapsed = time.time() - start_time
            eta = elapsed / epoch * (EPOCHS - epoch)
            
            if metrics:
                tqdm.write(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
                tqdm.write(f"  Health: sil_h={metrics['sil_h']:.4f}, sil_z={metrics['sil_z']:.4f}, kNN={metrics['knn_acc']:.4f}")
                breakdown = " | ".join([f"{k[:4]}:{v:.3f}" for k, v in metrics['per_class_h'].items()])
                tqdm.write(f"  Per-class (h): {breakdown}")
                tqdm.write(f"  ETA: {eta/60:.1f}m")
                
                per_class_str = ",".join([f"{metrics['per_class_h'].get(c, 0):.4f}" for c in class_names])
                with open(loss_csv, 'a') as f:
                    f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},{metrics['sil_h']:.4f},{metrics['sil_z']:.4f},{metrics['knn_acc']:.4f},{per_class_str}\n")
                    f.flush()
            else:
                tqdm.write(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e} | ETA: {eta/60:.1f}m")
                with open(loss_csv, 'a') as f:
                    f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},,,,{','*(len(class_names)-1)}\n")
        
        # Checkpoint
        if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
            ckpt_path = os.path.join(OUTPUT_DIR, f"BT_ckpt_epoch_{epoch:03d}.pth")
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save({
                "epoch": epoch,
                "encoder": state,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "loss": avg_loss,
                "best_sil_h": best_sil_h,
            }, ckpt_path)
            tqdm.write(f"  💾 Checkpoint: {ckpt_path}")
    
    # Final save
    final_path = os.path.join(OUTPUT_DIR, "SKANN_SSL_V3_Final.pth")
    state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    torch.save(state, final_path)
    
    total_time = time.time() - start_time
    
    print("\n" + "="*60)
    print("Training Complete")
    print("="*60)
    print(f"  Effective batch: {effective_batch}")
    print(f"  Final loss: {avg_loss:.4f}")
    print(f"  Best sil_h: {best_sil_h:.4f}")
    print(f"  Total time: {total_time/60:.1f} minutes")
    print(f"  Final model: {final_path}")
    print("="*60)
    
    return model


model = train()

## Cell 7: Extract Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings():
    log("Extracting embeddings (h and z)...")
    
    device = torch.device('cuda')
    
    m = model.module if hasattr(model, 'module') else model
    m.eval()
    
    files = sorted(glob.glob(os.path.join(TENSOR_DIR, 'tensor_*.npy')))
    
    pairing_df = pd.read_csv(PAIRING_PATH)
    clip_to_class = dict(zip(pairing_df['anchor_clip_id'], pairing_df['vessel_class']))
    class_to_id = {c: i for i, c in enumerate(sorted(pairing_df['vessel_class'].unique()))}
    
    h_embeddings, z_embeddings, labels = [], [], []
    
    for f in tqdm(files, desc="Extracting"):
        clip_id = int(os.path.basename(f).replace('tensor_', '').replace('.npy', ''))
        x = torch.from_numpy(np.load(f).flatten()).float().unsqueeze(0).to(device)
        
        h, z = m(x, return_features=True)
        h_embeddings.append(h.cpu().numpy().flatten())
        z_embeddings.append(z.cpu().numpy().flatten())
        labels.append(class_to_id.get(clip_to_class.get(clip_id, ''), -1))
    
    h_embeddings = np.array(h_embeddings)
    z_embeddings = np.array(z_embeddings)
    labels = np.array(labels)
    
    log(f"✅ Extracted: h={h_embeddings.shape}, z={z_embeddings.shape}")
    
    np.save(os.path.join(OUTPUT_DIR, 'embeddings_h.npy'), h_embeddings)
    np.save(os.path.join(OUTPUT_DIR, 'embeddings_z.npy'), z_embeddings)
    np.save(os.path.join(OUTPUT_DIR, 'labels.npy'), labels)
    
    return h_embeddings, z_embeddings, labels


h_embeddings, z_embeddings, labels = extract_embeddings()

## Cell 8: Final Metrics

In [ ]:
if h_embeddings is not None:
    mask = labels >= 0
    
    sil_h = silhouette_score(h_embeddings[mask], labels[mask], metric='cosine')
    sil_h_samples = silhouette_samples(h_embeddings[mask], labels[mask], metric='cosine')
    sil_z = silhouette_score(z_embeddings[mask], labels[mask], metric='cosine')
    
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn_scores = cross_val_score(knn, h_embeddings[mask], labels[mask], cv=5, scoring='accuracy')
    knn_acc = knn_scores.mean()
    
    print("\n" + "="*60)
    print("FINAL METRICS - BATCH=4 EXPERIMENT")
    print("="*60)
    print(f"  Effective batch: {BATCH_SIZE * torch.cuda.device_count()}")
    print(f"  Silhouette on h (backbone): {sil_h:.4f}  ← DEPLOYMENT METRIC")
    print(f"  Silhouette on z (projector): {sil_z:.4f}")
    print(f"  kNN accuracy on h: {knn_acc:.4f}")
    print("="*60)
    
    print(f"\nComparison:")
    print(f"  V1 baseline:    0.3997")
    print(f"  V2.1.0 (b=4):   0.8299")
    print(f"  V3 batch=11:    (Colab running)")
    print(f"  V3 batch=4→8:   {sil_h:.4f}  ← THIS RUN")
    
    pairing_df = pd.read_csv(PAIRING_PATH)
    classes = sorted(pairing_df['vessel_class'].unique())
    
    print(f"\nPer-class silhouette (h):")
    print("  KEY: fishing_vessel and small_craft should both be > 0.3")
    for i, cls in enumerate(classes):
        cls_mask = labels[mask] == i
        if cls_mask.sum() > 0:
            cls_sil = sil_h_samples[cls_mask].mean()
            status = "✅" if cls_sil > 0.5 else "⚠️" if cls_sil > 0 else "❌"
            print(f"  {cls}: {cls_sil:.4f} {status}")
    
    # Save
    with open(os.path.join(OUTPUT_DIR, 'final_metrics.txt'), 'w') as f:
        f.write(f"experiment: batch=4 per GPU (effective={BATCH_SIZE * torch.cuda.device_count()})\n")
        f.write(f"sil_h: {sil_h:.6f}\n")
        f.write(f"sil_z: {sil_z:.6f}\n")
        f.write(f"knn_acc: {knn_acc:.6f}\n")
        f.write(f"\nPer-class h:\n")
        for i, cls in enumerate(classes):
            cls_mask = labels[mask] == i
            if cls_mask.sum() > 0:
                f.write(f"{cls}: {sil_h_samples[cls_mask].mean():.6f}\n")

## Cell 9: Download Results

In [ ]:
print("\n" + "="*60)
print("Output Files (in /kaggle/working/output)")
print("="*60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f}: {os.path.getsize(fpath)/1e6:.2f} MB")
print("\nDownload from the Output tab on the right →")
print("="*60)